# 05 — Outlier Detection in Time Series: Rolling Stats, LOWESS & STL

**Companion notebook to Article 5 of the *Time Series Feature Engineering* series.**

📖 **Read the article:** [Medium](https://medium.com/@asidd24/outlier-detection-in-time-series-rolling-stats-lowess-stl-bc2af56a6276) · [LinkedIn Newsletter](https://www.linkedin.com/newsletters/time-series-engineered-7485187061080137728/)

---

> *"Every anomaly detector is a subtraction. You subtract what the series 'should' look like from what it actually is, and flag the residuals that exceed a threshold. The four methods in this article differ only in how they estimate 'should.'"*

---

### What This Notebook Covers:
1. **The Three Categories of Anomalies**: Point, Contextual, and Collective anomalies.
2. **Method 1 — Rolling Z-Score**: Local mean & standard deviation with strict leakage-safe `shift(1)`.
3. **Method 2 — Rolling IQR Bands**: Robust non-parametric quantile bands (Tukey's IQR rule).
4. **Method 3 — LOWESS Residual Detection**: Locally-weighted regression tracking non-linear smooth trend.
5. **Method 4 — STL Residual Detection**: Seasonal-Trend decomposition with LOESS and `robust=True`.
6. **Head-to-Head Method Comparison**: Evaluating detection coverage, precision, and false positives.
7. **The Production Pattern — Detector as a Feature**: Engineering anomaly features (`stl_residual`, `residual_volatility`, `days_since_last_anomaly`) for downstream forecasting models.

## 1. Setup & Imports

In [ ]:
import sys
sys.path.append('..')  # Allow importing from src/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.seasonal import STL
from statsmodels.nonparametric.smoothers_lowess import lowess

from src.data import generate_daily_sales
from src.features import (
    rolling_zscore,
    rolling_iqr_bands,
    lowess_anomalies,
    stl_anomalies,
    add_anomaly_features,
)

# Plotting styles
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 11
np.random.seed(42)

## 2. Generating Synthetic Time Series & Injected Anomalies

We generate 2 years (730 days) of daily retail sales with trend, weekly seasonality, and realistic noise.

We inject three canonical categories of anomalies:
1. **Point anomalies**: Isolated, single-day spikes or dips (e.g., flash sales, sudden stock-outs).
2. **Contextual anomalies**: Values that look normal in magnitude but are abnormal given the day-of-week context.
3. **Collective anomalies**: A continuous run of abnormal days (e.g., a 10-day promotional event or supply chain disruption).

In [ ]:
# Generate clean reference daily sales series
clean_sales = generate_daily_sales(n_days=730, seed=42)
sales = clean_sales.copy()

# 1. Inject Point Anomalies (single extreme points)
point_indices = [60, 185, 290, 420, 540, 670]
point_magnitudes = [45, -35, 55, -40, 60, -45]
for idx, mag in zip(point_indices, point_magnitudes):
    sales.iloc[idx] += mag

# 2. Inject Contextual Anomaly (abnormal level during seasonal trough)
contextual_range = slice(330, 340)
sales.iloc[contextual_range] += 25

# 3. Inject Collective Anomaly (sustained elevated run in November)
collective_range = slice(500, 514)
sales.iloc[collective_range] += 30

print(f"Total Observations: {len(sales)}")
print(f"Point Anomalies injected at positions: {point_indices}")
print(f"Contextual Anomaly injected at index range: 330:340")
print(f"Collective Anomaly injected at index range: 500:514")

In [ ]:
# Figure 1: Visualizing the Three Anomaly Categories
fig, ax = plt.subplots(figsize=(15, 5))

ax.plot(sales.index, sales.values, color='#1f77b4', linewidth=1.2, label='Observed Sales Series')
ax.scatter(sales.index[point_indices], sales.iloc[point_indices], color='#d62728', s=60, zorder=5, label='Point Anomalies (Spikes/Dips)')
ax.axvspan(sales.index[330], sales.index[339], color='#ff7f0e', alpha=0.25, label='Contextual Anomaly (Shift in Trough)')
ax.axvspan(sales.index[500], sales.index[513], color='#9467bd', alpha=0.25, label='Collective Anomaly (14-Day Shift)')

ax.set_title('Figure 1 — The Three Categories of Time Series Anomalies', fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Daily Sales Units')
ax.legend(loc='upper left', frameon=True)
plt.tight_layout()
plt.show()

## 3. Method 1 — Rolling Z-Score

The rolling z-score is the workhorse of operational monitoring.

$$\mu_t = \text{mean}(y_{t-W : t-1}), \quad \sigma_t = \text{std}(y_{t-W : t-1})$$
$$z_t = \frac{y_t - \mu_t}{\sigma_t}$$

> **Crucial Leakage Rule:** We apply `shift(1)` before the rolling calculation so that observation $y_t$ never contaminates its own baseline mean or standard deviation.

In [ ]:
# Compute rolling z-scores with a 28-day window
z_scores = rolling_zscore(sales, window=28)
z_threshold = 3.0
z_anomalies = sales[z_scores.abs() > z_threshold]

print(f"Rolling Z-Score Detections (|z| > {z_threshold}): {len(z_anomalies)}")

In [ ]:
# Figure 2: Rolling Z-Score Detection (2-Panel Plot)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 7), sharex=True, gridspec_kw={'height_ratios': [2, 1.2]})

# Top panel: Sales series and flagged anomalies
ax1.plot(sales.index, sales.values, color='#1f77b4', linewidth=1.2, label='Sales')
ax1.scatter(z_anomalies.index, z_anomalies.values, color='#d62728', s=50, zorder=5, label=f'Detected Anomalies (|z| > {z_threshold})')
ax1.set_title('Figure 2 — Rolling Z-Score Anomaly Detection (Window = 28 Days)', fontweight='bold')
ax1.set_ylabel('Daily Sales Units')
ax1.legend(loc='upper left')

# Bottom panel: Z-score profile and ±3 sigma bands
ax2.plot(z_scores.index, z_scores.values, color='#333333', linewidth=1, label='Rolling Z-Score')
ax2.axhline(z_threshold, color='#d62728', linestyle='--', linewidth=1.2, label=f'+{z_threshold}σ Threshold')
ax2.axhline(-z_threshold, color='#d62728', linestyle='--', linewidth=1.2, label=f'-{z_threshold}σ Threshold')
ax2.axhline(0, color='gray', linestyle=':', alpha=0.6)
ax2.fill_between(z_scores.index, -z_threshold, z_threshold, color='gray', alpha=0.15)
ax2.set_ylabel('Z-Score (σ)')
ax2.set_xlabel('Date')
ax2.set_ylim(-5, 6)
ax2.legend(loc='upper left')

plt.tight_layout()
plt.show()

## 4. Method 2 — Rolling IQR Bands

When time series residuals have heavy tails or sudden volatility, sample variance $\sigma$ becomes inflated by extreme points. **Rolling IQR** replaces mean and standard deviation with quantiles:

$$\text{IQR}_t = Q_{3, t} - Q_{1, t}$$
$$\text{Upper}_t = Q_{3, t} + k \times \text{IQR}_t, \quad \text{Lower}_t = Q_{1, t} - k \times \text{IQR}_t$$

We use the classical Tukey multiplier $k = 1.5$.

In [ ]:
# Compute rolling IQR bands (28-day window, k=1.5)
lower_band, upper_band = rolling_iqr_bands(sales, window=28, k=1.5)
iqr_anomalies = sales[(sales > upper_band) | (sales < lower_band)]

print(f"Rolling IQR Detections: {len(iqr_anomalies)}")

In [ ]:
# Figure 3: Rolling IQR Bands with Adaptive Confidence Envelope
fig, ax = plt.subplots(figsize=(15, 5))

ax.plot(sales.index, sales.values, color='#1f77b4', linewidth=1.2, label='Sales')
ax.fill_between(sales.index, lower_band, upper_band, color='#2ca02c', alpha=0.2, label='Rolling IQR Acceptance Band (Q1-1.5IQR to Q3+1.5IQR)')
ax.plot(sales.index, upper_band, color='#2ca02c', linestyle='--', linewidth=0.9)
ax.plot(sales.index, lower_band, color='#2ca02c', linestyle='--', linewidth=0.9)
ax.scatter(iqr_anomalies.index, iqr_anomalies.values, color='#d62728', s=50, zorder=5, label='Detected Anomalies (Outside IQR Band)')

ax.set_title('Figure 3 — Rolling IQR Anomaly Detection (Adaptive Robust Quantile Envelope)', fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Daily Sales Units')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

## 5. Method 3 — LOWESS Residual Detection

Rolling methods assume that "normal" is stationary within each rolling window. For series with strong non-linear trend curves, **LOWESS** (Locally-Weighted Scatterplot Smoothing) fits a flexible nonparametric smooth trend and evaluates residuals:

$$e_t = y_t - \hat{y}_{\text{LOWESS}, t}$$
$$\text{Flag if } |e_t - \bar{e}| > 3 \times \sigma_e$$

In [ ]:
# Fit LOWESS and extract anomalies (frac=0.05, 3-sigma)
lowess_res = lowess_anomalies(sales, frac=0.05, threshold_sigma=3.0)

# Compute LOWESS fit for visualization
smoothed = lowess(sales.values, np.arange(len(sales)), frac=0.05, return_sorted=False)
lowess_fit = pd.Series(smoothed, index=sales.index)
residuals_lowess = sales - lowess_fit

print(f"LOWESS Residual Detections: {len(lowess_res)}")

In [ ]:
# Figure 4: LOWESS Fit & Residuals
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 7), sharex=True, gridspec_kw={'height_ratios': [2, 1.2]})

# Top: Raw Series & LOWESS Fit
ax1.plot(sales.index, sales.values, color='#1f77b4', alpha=0.7, label='Observed Sales')
ax1.plot(lowess_fit.index, lowess_fit.values, color='#ff7f0e', linewidth=2, label='LOWESS Non-parametric Trend (frac=0.05)')
ax1.scatter(lowess_res.index, sales.loc[lowess_res.index], color='#d62728', s=50, zorder=5, label='LOWESS Detected Anomalies')
ax1.set_title('Figure 4 — LOWESS Non-parametric Smoothing & Residual Anomaly Detection', fontweight='bold')
ax1.set_ylabel('Sales Units')
ax1.legend(loc='upper left')

# Bottom: Residuals
std_l = residuals_lowess.std()
ax2.plot(residuals_lowess.index, residuals_lowess.values, color='#333333', linewidth=1, label='LOWESS Residuals ($y - \hat{y}$)')
ax2.axhline(3 * std_l, color='#d62728', linestyle='--', label='±3σ Residual Threshold')
ax2.axhline(-3 * std_l, color='#d62728', linestyle='--')
ax2.fill_between(residuals_lowess.index, -3 * std_l, 3 * std_l, color='gray', alpha=0.15)
ax2.set_ylabel('Residual (Units)')
ax2.set_xlabel('Date')
ax2.legend(loc='upper left')

plt.tight_layout()
plt.show()

## 6. Method 4 — STL Residual Detection

**STL (Seasonal-Trend Decomposition using LOESS)** is the modern standard for structured business time series.

By decomposing $y_t = T_t + S_t + R_t$ with `robust=True`, STL iteratively downweights outliers when estimating the trend $T_t$ and seasonal cycle $S_t$. The remainder $R_t$ contains only pure idiosyncratic variations and structural anomalies.

In [ ]:
# STL Decomposition with robust fitting
stl_fit = STL(sales, period=7, robust=True).fit()
stl_res_series = stl_fit.resid

stl_detected = stl_anomalies(sales, period=7, threshold_sigma=3.0)
print(f"STL Robust Residual Detections: {len(stl_detected)}")

In [ ]:
# Figure 5: STL Residual-Based Anomaly Detection
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 7), sharex=True, gridspec_kw={'height_ratios': [2, 1.2]})

# Top panel: Observed series & Baseline (Trend + Seasonal)
baseline = stl_fit.trend + stl_fit.seasonal
ax1.plot(sales.index, sales.values, color='#1f77b4', alpha=0.7, label='Observed Sales')
ax1.plot(baseline.index, baseline.values, color='#2ca02c', linewidth=1.5, label='STL Expected Baseline (Trend + Seasonality)')
ax1.scatter(stl_detected.index, sales.loc[stl_detected.index], color='#d62728', s=50, zorder=5, label='STL Detected Anomalies (|R| > 3σ)')
ax1.set_title('Figure 5 — STL Decomposition Residual Anomaly Detection (Robust LOESS)', fontweight='bold')
ax1.set_ylabel('Sales Units')
ax1.legend(loc='upper left')

# Bottom panel: Residual component
std_stl = stl_res_series.std()
ax2.plot(stl_res_series.index, stl_res_series.values, color='#333333', linewidth=1, label='STL Remainder ($R_t$)')
ax2.axhline(3 * std_stl, color='#d62728', linestyle='--', label='±3σ Threshold')
ax2.axhline(-3 * std_stl, color='#d62728', linestyle='--')
ax2.fill_between(stl_res_series.index, -3 * std_stl, 3 * std_stl, color='gray', alpha=0.15)
ax2.set_ylabel('Remainder (Units)')
ax2.set_xlabel('Date')
ax2.legend(loc='upper left')

plt.tight_layout()
plt.show()

## 7. Head-to-Head Comparison Across All Four Methods

Let's compare what each detector catches across all injected point, contextual, and collective anomalies.

In [ ]:
# Figure 6: Head-to-Head Comparison Plot
fig, ax = plt.subplots(figsize=(15, 6))

# Plot background series
ax.plot(sales.index, sales.values, color='lightgray', linewidth=1.2, label='Sales Series')

# True injected point anomalies (black stars)
ax.scatter(sales.index[point_indices], sales.iloc[point_indices], color='black', marker='*', s=140, zorder=6, label='True Injected Point Anomalies')

# Detection flags rendered as horizontal tick tracks
y_min, y_max = sales.min(), sales.max()
y_track_base = y_max + 10

ax.scatter(z_anomalies.index, [y_track_base + 12] * len(z_anomalies), color='#1f77b4', marker='|', s=100, linewidth=2, label='Rolling Z-Score Flags')
ax.scatter(iqr_anomalies.index, [y_track_base + 8] * len(iqr_anomalies), color='#2ca02c', marker='|', s=100, linewidth=2, label='Rolling IQR Flags')
ax.scatter(lowess_res.index, [y_track_base + 4] * len(lowess_res), color='#ff7f0e', marker='|', s=100, linewidth=2, label='LOWESS Residual Flags')
ax.scatter(stl_detected.index, [y_track_base] * len(stl_detected), color='#d62728', marker='|', s=100, linewidth=2, label='STL Residual Flags')

# Highlight contextual and collective regions
ax.axvspan(sales.index[330], sales.index[339], color='#ff7f0e', alpha=0.15, label='Contextual Anomaly Region')
ax.axvspan(sales.index[500], sales.index[513], color='#9467bd', alpha=0.15, label='Collective Anomaly Region')

ax.set_title('Figure 6 — Comprehensive Method Comparison on the Same Time Series', fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Sales Units')
ax.legend(loc='lower left', ncol=2, frameon=True)
ax.set_ylim(y_min - 10, y_track_base + 20)
plt.tight_layout()
plt.show()

In [ ]:
# Compare detection performance metrics
true_point_dates = set(sales.index[point_indices])

summary = pd.DataFrame({
    'Method': ['Rolling Z-Score', 'Rolling IQR', 'LOWESS Residuals', 'STL Residuals'],
    'Total Flags': [len(z_anomalies), len(iqr_anomalies), len(lowess_res), len(stl_detected)],
    'Point Anomalies Caught (out of 6)': [
        len(true_point_dates.intersection(z_anomalies.index)),
        len(true_point_dates.intersection(iqr_anomalies.index)),
        len(true_point_dates.intersection(lowess_res.index)),
        len(true_point_dates.intersection(stl_detected.index)),
    ],
    'Catches Collective Run': ['No', 'No', 'Partial', 'Yes (Full Cluster)'],
    'Seasonal Awareness': ['Low', 'Low', 'Moderate', 'High'],
})

summary

## 8. The Production Pattern — Detector as a Feature

Rather than using anomaly detection only as an alert system, in machine learning forecasting pipelines we engineer **anomaly-derived features** to inform models about recent regime instability:

- `stl_residual`: Raw residual magnitude.
- `residual_volatility`: Rolling standard deviation of past residuals (`shift(1)` safe).
- `is_extreme_residual`: Binary flag for extreme deviations.
- `days_since_last_anomaly`: Days elapsed since the last detected anomaly.

In [ ]:
# Generate feature set using src.features.add_anomaly_features
anomaly_features = add_anomaly_features(sales, period=7, threshold_sigma=3.0)

# Combine with target
feature_matrix = pd.concat([sales.rename('sales'), anomaly_features], axis=1)
feature_matrix.tail(10)

In [ ]:
# Visualize the engineered anomaly features
fig, axes = plt.subplots(3, 1, figsize=(15, 8), sharex=True)

# Panel 1: Residual Volatility
axes[0].plot(feature_matrix.index, feature_matrix['residual_volatility'], color='#e377c2', linewidth=1.2)
axes[0].set_ylabel('Residual Volatility (14d σ)')
axes[0].set_title('Engineered Feature: Residual Volatility (Detects Structural Turbulence)', fontweight='bold')

# Panel 2: Days since last anomaly
axes[1].plot(feature_matrix.index, feature_matrix['days_since_last_anomaly'], color='#17becf', linewidth=1.2)
axes[1].set_ylabel('Days Elapsed')
axes[1].set_title('Engineered Feature: Days Since Last Anomaly (Decay Signal)', fontweight='bold')

# Panel 3: Extreme Residual Flag
axes[2].stem(feature_matrix.index, feature_matrix['is_extreme_residual'], linefmt='#d62728-', markerfmt='ro', basefmt='k-')
axes[2].set_ylabel('Is Extreme (0/1)')
axes[2].set_title('Engineered Feature: Binary Extreme Residual Indicator', fontweight='bold')
axes[2].set_xlabel('Date')

plt.tight_layout()
plt.show()

## 9. Decision Framework & Summary

| Situation | Recommended Method | Rationale |
|---|---|---|
| **Operational monitoring, roughly stationary** | `Rolling z-score` | Fast, simple, standard 3-sigma monitoring |
| **Fat-tailed distributions, volatile metrics** | `Rolling IQR bands` | Quantile-based, immune to variance distortion |
| **Smooth non-linear trend, weak seasonality** | `LOWESS residuals` | Non-parametric curve fitting isolating trend |
| **Trend + strong seasonality (retail, web, energy)** | `STL residuals` | Modern default; subtracts seasonal cycles cleanly |
| **High-stakes production alerts (low false alarm)** | `STL AND Rolling IQR` | Ensemble consensus rule |
| **Forecasting model feature inputs** | `add_anomaly_features` | Supplies regime shifts and shock proximity to ML models |

---

### Key Takeaways:
1. **Every anomaly detector is a subtraction**: The only difference between methods is how they model the expected baseline.
2. **Respect temporal boundaries**: Always compute rolling statistics on shifted data (`shift(1)`) and refit STL/LOWESS within walk-forward CV splits to prevent future leakage.
3. **Anomalies as features**: Instead of merely discarding or capping outliers, feeding residual volatility and anomaly recency into tree models improves forecast resilience after turbulent periods.

---

**Next in the series:** Article 6 — *"Lag Features: Teaching Your Model to Remember the Past"*.